In [1]:
# Celda 1
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Configuración visual para los gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Celda 2
gallery_path = "../data/gallery/gallery_embeddings.pkl" # Ajusta el path según tu estructura

if not os.path.exists(gallery_path):
    raise FileNotFoundError(f"No se encontró {gallery_path}. Ejecuta build_gallery() primero.")

with open(gallery_path, 'rb') as f:
    gallery = pickle.load(f)

X = []
y = []

# Aplanamos el diccionario
for identidad, embs in gallery.items():
    for emb in embs:
        X.append(emb)
        y.append(identidad)

X = np.array(X)
y = np.array(y)

print(f"Total de embeddings cargados: {X.shape[0]}")
print(f"Dimensiones del embedding original: {X.shape[1]}")
print(f"Total de identidades únicas: {len(np.unique(y))}")

In [ ]:
# Celda 3
# 1. PCA: Busca las direcciones de mayor varianza (lineal)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

# 2. t-SNE: Preserva la estructura local de los vecinos (no lineal)
# Perplexity suele ajustarse entre 5 y 50 dependiendo de la cantidad de datos.
# Como tenemos pocas fotos por persona (3-5), usamos una perplexity baja.
tsne = TSNE(n_components=2, perplexity=10, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X)

In [ ]:
# Celda 4
identidades_unicas = np.unique(y)
colores = plt.cm.tab10(np.linspace(0, 1, len(identidades_unicas)))

fig, (ax1, ax2) = plt.subplots(1, 2)

# Gráfico PCA
for i, identidad in enumerate(identidades_unicas):
    mask = y == identidad
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], color=colores[i], label=identidad, alpha=0.7)
ax1.set_title('Proyección con PCA')
ax1.set_xlabel('Componente Principal 1')
ax1.set_ylabel('Componente Principal 2')

# Gráfico t-SNE
for i, identidad in enumerate(identidades_unicas):
    mask = y == identidad
    ax2.scatter(X_tsne[mask, 0], X_tsne[mask, 1], color=colores[i], label=identidad, alpha=0.7)
ax2.set_title('Proyección con t-SNE')
ax2.set_xlabel('Dimensión t-SNE 1')
ax2.set_ylabel('Dimensión t-SNE 2')

# Leyenda global
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=min(len(identidades_unicas), 5), bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

INTERPRETACION DE LOS GRAFICOS

In [ ]:
# Celda final: Evaluación de Métricas y Registro en Weights & Biases (WandB)
import os
import pickle
from src.reid.embeddings import build_gallery
from src.reid.metrics import evaluate_reid
from src.tracking.wandb_utils import init_run, log_epoch

# 1. Asegurar la construcción de la galería (usa la caché si ya existe)
# (Coordina con Jose qué carpeta exacta usarán para query/probe set)
gallery_dir = "../data/gallery" 
gallery = build_gallery(gallery_dir)

# NOTA: Para la evaluación necesitas un 'query_set' (o probe set).
# Si aún no lo han separado en una carpeta distinta, puedes estructurarlo igual que la galería
# o usar un subset de imágenes de prueba de la misma manera: {identidad: [emb, emb, ...]}.
# Por ahora, asumimos que tienes un diccionario 'query_set' con tus imágenes de consulta.
# Ejemplo simulado para que el código no falle:
query_set = gallery  # Reemplazar por el query_set real cuando esté aislado

# 2. Calcular las métricas con tu variante custom de mAP re-ID
resultados = evaluate_reid(query_set, gallery)

print("\n=== RENDIMIENTO DE RE-IDENTIFICACIÓN ===")
print(f"Top-1 Accuracy: {resultados['top1']:.4f}")
print(f"Top-5 Accuracy: {resultados['top5']:.4f}")
print(f"mAP (Variante Re-ID): {resultados['map']:.4f}")
print("========================================\n")

# 3. Inicializar el experimento en WandB consumiendo el contrato de Nicole
print("Inicializando sesión en Weights & Biases...")
init_run(
    name="evaluacion_reid_arcface_v1", 
    config={
        "extractor_features": "DeepFace/ArcFace",
        "metrica_distancia": "cosine",
        "fase": "2A - Medición Re-ID"
    }
)

# 4. Registrar los resultados en la plataforma para cumplir con la Fase 5
print("Logueando métricas en WandB...")
log_epoch(metrics=resultados, step=1)

print("¡Proceso de evaluación y registro completado con éxito!")